# 기억 탐험가 V2

ChromaDB + ConversationBufferMemory 기반 대화형 일기 탐험 챗봇

- **검색**: Retriever.ipynb 패턴 — 의도 분석 → enriched_query(키워드+감정) → 벡터 유사도 검색
- **메모리**: `ConversationBufferMemory`로 대화 문맥 유지

- 아직 기능적으로 좀 더 고민을 해봐야 될 것 같음...

In [48]:
import os, uuid, yaml
from datetime import date, datetime, timedelta
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.memory import ConversationBufferMemory
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

In [49]:
os.environ["ANONYMIZED_TELEMETRY"] = "False"

target_user = '김부장'
CHROMA_PATH = f"./data/{target_user}/chroma_db"
today = '2026-03-27'

EMOTIONS = ["기쁨", "놀라움", "두려움", "분노", "불쾌함", "설렘", "슬픔", "평범함"]

In [ ]:
# langchain_chroma로 벡터스토어 초기화
# filter는 Python 사이드에서만 처리 — langchain_chroma에 where 절대 전달 안 함
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    collection_name="langchain",
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH,
)

print(f"DB 로드 완료 — 저장된 일기 수: {vectorstore._collection.count()}")


def chroma_search(query_text: str, k: int = 20) -> List[dict]:
    """벡터 유사도 검색. filter 없이 전체 탐색 후 Python-side 날짜 필터링."""
    total = vectorstore._collection.count()
    if total == 0:
        return []
    n = min(k, total)

    results = vectorstore.similarity_search_with_score(query_text, k=n)

    return [
        {
            "content": doc.page_content,
            "metadata": doc.metadata,
            "distance": score,
        }
        for doc, score in results
    ]


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


DB 로드 완료 — 저장된 일기 수: 107


In [51]:
def add_diary_entry(date_str: str, emotion: str, diary: str) -> None:
    """새 일기를 벡터스토어에 추가."""
    doc = Document(
        page_content=diary,
        metadata={
            "date": date_str,
            "emotion": emotion,
            "source": f"diary_{date_str}",
        },
    )
    vectorstore.add_documents([doc])
    print(f"[{date_str}] 일기 추가 완료 — 현재 총 {vectorstore._collection.count()}개")


In [ ]:
class SearchIntent(BaseModel):
    refined_query: str = Field(description="검색에 사용할 핵심 사건·인물·장소 키워드 (사용자의 불확실한 기억 표현 '~인 것 같아', '~였던 것 같아'는 제외)")
    target_date: Optional[str] = Field(default=None, description="특정 단일 날짜 (YYYY-MM-DD)")
    start_date: Optional[str] = Field(default=None, description="날짜 범위 시작 (YYYY-MM-DD)")
    end_date: Optional[str] = Field(default=None, description="날짜 범위 종료 (YYYY-MM-DD)")
    is_temporal_navigation: bool = Field(default=False, description="'다음 날', '전날' 등 상대적 시간 이동 여부")
    navigation_days: int = Field(default=0, description="이동 일수 (다음 날: 1, 전날: -1, 이틀 뒤: 2)")
    target_emotions: List[str] = Field(default=[], description="유사 감정 확장 리스트")
    is_date_unknown: bool = Field(default=False, description="날짜를 명확히 모르고 '언제였지?', '정확히 기억이 안 나는데', '언제 있었더라' 등으로 물을 때 True")


# yaml 파일에서 프롬프트 로드
with open("./prompts/memory_explorer.yaml", "r", encoding="utf-8") as f:
    prompts_config = yaml.safe_load(f)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 의도 분석 프롬프트
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts_config["analysis_system_prompt"]),
    ("user", prompts_config["analysis_human_template"])
])

# 단일 응답 프롬프트
response_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts_config["response_system_prompt"]),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", prompts_config["response_human_template"])
])

# 다중 응답 프롬프트
multi_response_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts_config["multi_response_system_prompt"]),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", prompts_config["multi_response_human_template"])
])

structured_llm = llm.with_structured_output(SearchIntent, method="function_calling")
analyzer = analysis_prompt.partial(today=today) | structured_llm

# 지시어 하드 코딩
DEICTIC_WORDS = ["거기서", "거기에서", "그때", "그날", "그 날", "그 자리", "그 곳", "거기", "그곳에서"]

In [53]:
# 대화 메모리 - yaml에서 로드한 프롬프트 사용
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

In [ ]:
state = {"last_date": None, "last_summary": "없음", "candidates": []}


def memory_explorer_chat(query: str, debug: bool = False) -> str:
    # (1) 의도 분석 - 5주차 Retriever의 내용
    intent = analyzer.invoke({
        "query": query,
        "last_date": state["last_date"] or "없음",
        "last_summary": state["last_summary"],
    })

    # (2) 날짜 결정 - 이전 대화의 내용을 바탕으로 현재의 날짜를 결정
    search_date = intent.target_date
    if any(w in query for w in DEICTIC_WORDS) and state["last_date"]:
        search_date = state["last_date"]
    elif intent.is_temporal_navigation and state["last_date"]:
        base = datetime.strptime(state["last_date"], "%Y-%m-%d")
        search_date = (base + timedelta(days=intent.navigation_days)).strftime("%Y-%m-%d")

    # (3) enriched query 생성 - 5주차 Retriever의 내용
    emotions_str = " ".join(intent.target_emotions)
    enriched_query = f"{intent.refined_query} {emotions_str}".strip()

    # 의도, 날짜 탐색이 제대로 진행되고 있는지 확인하기 위한 Debuging 코드
    if debug:
        print(f"[DEBUG] refined='{intent.refined_query}' | search_date={search_date} | date_unknown={intent.is_date_unknown}")

    # (4) 벡터 유사도 검색 - 5주차 Retriever의 내용
    candidates = chroma_search(enriched_query, k=20)

    # (5) Python-side 날짜 필터링 (- DB filter를 사용하지 않고 python 단계에서 사용.)
    # chroma DB에서 where 사용 시 자꾸 에러가 발생 -> 해당 문제의 원인 탐색 도중 잘 모르겠어서 -> 일단은 파이썬에서 처리하는 방식 채택.
    if search_date:
        docs = [c for c in candidates if c["metadata"].get("date") == search_date]
    elif intent.start_date and intent.end_date:
        docs = [c for c in candidates
                if intent.start_date <= c["metadata"].get("date", "") <= intent.end_date]
    else:
        docs = candidates

    if not docs:
        if search_date:
            return f"[{search_date}] 날짜에 해당하는 일기 기록을 찾지 못했습니다."
        return "해당 기억에 대한 기록을 찾지 못했습니다."

    chat_history = memory.load_memory_variables({})["chat_history"]

    # (6) 날짜 불명확 → top-3 후보 제시(multi_response_prompt 사용)
    if intent.is_date_unknown and not search_date and len(docs) >= 2:
        top_docs = docs[:3]
        state["candidates"] = [d["metadata"].get("date") for d in top_docs]
        # 가장 유사도 높은 것을 잠정 last_date로 설정
        state["last_date"] = top_docs[0]["metadata"].get("date")
        state["last_summary"] = top_docs[0]["content"][:150]

        context_parts = [
            f"[후보 {i}] [{d['metadata'].get('date')} / {d['metadata'].get('emotion')}]\n{d['content']}"
            for i, d in enumerate(top_docs, 1)
        ]
        context = "\n\n".join(context_parts)

        chain = multi_response_prompt | llm | StrOutputParser()
        answer = chain.invoke({"query": query, "context": context, "chat_history": chat_history})
        memory.save_context({"input": query}, {"output": answer})

        dates_str = " / ".join(state["candidates"])
        return f"[후보: {dates_str}]\n{answer}"

    # (7) 단일 결과 (날짜가 명확하고, 해당 내용을 일기에서 찾을 수 있을 경우)
    top = docs[0]
    state["last_date"] = top["metadata"].get("date")
    state["last_summary"] = top["content"][:150]
    state["candidates"] = []
    context = f"[{top['metadata'].get('date')} / {top['metadata'].get('emotion')}]\n{top['content']}"

    chain = response_prompt | llm | StrOutputParser()
    answer = chain.invoke({"query": query, "context": context, "chat_history": chat_history})
    memory.save_context({"input": query}, {"output": answer})

    return f"[{top['metadata'].get('date')}]\n{answer}"


## 챗봇 테스트

In [ ]:
memory.clear()
state["last_date"] = None
state["last_summary"] = "없음"

print(memory_explorer_chat("작년 12월에 딸이랑 가구 보러 갔던 거 기억나?"))

[2025-12-13]
혹시 아내와 함께 가셨던 건 아닐까요? 가구 박람회에서 예쁜 인테리어 소품들을 구경하며 집을 꾸미고 싶은 마음이 커졌던 기억이 나네요. 그날 어떤 소품을 구입하셨는지 기억하시나요?


In [58]:
print(memory_explorer_chat("작은 인테리어용 오르골을 샀어.", debug=True))


[DEBUG] refined='인테리어용 오르골' | search_date=None | date_unknown=False
[2026-01-14]
그 내용은 일기에 남아있지 않네요. 소중한 기억이니 일기에 추가해두시는 건 어떨까요? 그날 오르골을 구입한 후에 다른 가구나 소품도 구경하셨나요?


In [59]:
print(memory_explorer_chat("여러 가지 가구나 소품들도 구경했지, 정확히 뭘 봤는지는 기억에 나지 않지만..."))

[후보: 2025-12-13 / 2026-02-07 / 2026-01-30]
비슷한 기억이 몇 가지 있어요.  
① [2025-12-13] 아내와 가구 박람회에 다녀와서 예쁜 인테리어 소품들을 구경하며 설레었던 날.  
② [2026-02-07] 아내와 가전 매장에서 공기청정기를 고르며 기대에 부풀었던 날.  
③ [2026-01-30] 거래처의 무책임한 태도로 인해 분노하며 힘든 하루를 보냈던 날.  
어떤 날이 맞을까요?


In [60]:
print(memory_explorer_chat("1번 기억에서 말이야."))


[2025-12-13]
그날 아내와 함께 가구 박람회에 다녀오셨군요! 예쁜 인테리어 소품들을 구경하며 집을 꾸미고 싶은 마음이 커졌던 기억이 정말 설레네요. 그날 구입한 작은 소품이 집 분위기를 바꿔주었다니, 어떤 변화가 있었는지 궁금하네요. 그 소품은 어떤 느낌이었나요?


In [ ]:
def run_memory_explorer():
    """대화형 기억 탐험가를 실행합니다."""
    memory.clear()
    state["last_date"] = None

    print("=== 기억 탐험가 V2 ===")
    print("과거의 일기를 함께 탐험해봐요. 종료하려면 'q'를 입력하세요.\n")

    while True:
        query = input("you > ").strip()
        if query.lower() in ("q", "quit", "exit"):
            print("대화를 종료합니다.")
            break
        if not query:
            continue
        response = memory_explorer_chat(query)
        print(f"\nassistant > {response}\n")

# run_memory_explorer()